# Impedance ML Engineering — Session 08
## Understand tensors, gradients, and optimization
**Core: 60 minutes · Neural networks · Optional extension: 20–40 minutes**

## Why this matters
What exactly does a neural network compute and learn from impedance features?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Reason about tensor shapes and parameter counts.
- Compute a regression loss and inspect autograd.
- Verify a simple gradient numerically.
- Debug broadcasting and gradient accumulation.

## R-to-Python / prerequisite bridge
NumPy arrays → torch tensors with explicit dtype; R recycled vectors differ from broadcasting; PyTorch gradients accumulate unless cleared.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
An affine layer computes xWᵀ+b; hidden activations make a composition nonlinear. Training computes a scalar loss, differentiates it with respect to parameters by the chain rule, and updates those parameters with an optimizer. Autograd records tensor operations; gradients accumulate on leaf tensors until cleared.

Broadcasting can make incorrect shapes execute instead of fail. Explicit shape equality before a loss is a valuable guard. Gradient descent is local optimization of a training objective, not a guarantee of generalization.

Imports: `import torch`, `from torch import nn`. `torch.as_tensor(array,dtype=torch.float32)`, `nn.Linear(in_features,out_features)`, `loss.backward()`, `parameter.grad`, and `optimizer.zero_grad(set_to_none=True)` are the core APIs. Keep numerically sensitive finite-difference checks in float64.

## Task 1 — Convert one training batch (5 min)
Select 16 training rows using only two numeric features: current diagonal mean and planned horizon. Convert to float32 tensors with `torch.as_tensor`. Make y shape (16, 1). Predict shapes and verify no NaNs. Keep target and inputs on CPU for this lesson; no GPU is needed.

## Task 2 — Inspect an affine layer (6 min)
Create `nn.Linear(2,1)` after `torch.manual_seed(42)`. Inspect weight/bias shapes and calculate parameter count by hand before `.numel()`. Forward the batch and verify (16, 1). Explain why sample count is not part of parameter count.

## Task 3 — Calculate and interpret a loss (6 min)
Compute mean squared error with `nn.MSELoss()`. Manually recompute mean squared residual and verify equality. State units: squared kΩ. Contrast the training loss with reported MAE in kΩ; optimization objective and reporting metric need not be identical.

## Task 4 — Trace autograd (8 min)
Call loss.backward and inspect parameter `.grad`. Create a scalar tensor requiring gradients and evaluate a simple expression of your choice; predict the derivative before `.backward()`. Inspect `requires_grad` and `.grad_fn`. Explain why converting through NumPy can sever the computation graph.

## Task 5 — Check a gradient with finite differences (8 min)
On an unrelated one-parameter function f(w)=(w*2-3)^2, compare autograd with `(f(w+eps)-f(w-eps))/(2*eps)` using float64, w=0.7, eps=1e-5. Explain why the epsilon and numerical dtype matter. Acceptance: close agreement; the example is separate from the matrix prediction task.

## Task 6 — Take controlled optimizer steps (8 min)
Instantiate `torch.optim.SGD(model.parameters(),lr=0.001)`. For ten steps: zero gradients, forward, loss, backward, step. Log loss and one parameter; use `zero_grad(set_to_none=True)`. Compare before/after values. Explain the learning-rate role and why a decrease in training loss is not validation evidence.

## Task 7 — Debug silent shape errors (7 min)
Audit `prediction.shape==(16,1)` with `target.shape==(16,)` passed to MSELoss: predict the broadcast residual shape and why the result is wrong. Fix target shape and assert equality. Then explain why two backward calls without zeroing gradients alter the next update.

## Task 8 — Add a nonlinear hidden layer (4 min)
Create Linear(2, 8) → ReLU → Linear(8, 1). Predict output shape and parameter count before running. Describe how the nonlinear model differs from composing two linear layers without an activation. No predictive contest yet: focus on mechanics.

## Retrieval practice
Explain the input, output, weight, and gradient shapes of Linear(144, 16) for a batch of five flattened matrices.

## Professional reasoning
1. What could go wrong if integers are accidentally used for a regression tensor?
2. Why is calling `.item()` appropriate for logging but not differentiable loss composition?
3. Why can optimization succeed while generalization fails?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Plot an activation and its gradient over a small input grid; compare ReLU with tanh and describe saturation.

## API references
[PyTorch training basics](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html) · [Conv2d reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)